In [1]:
# 0. Setup (Colab 런타임에서 실행)
import os, subprocess, sys

REPO_URL = "https://github.com/limJhyeok/vopt-learning.git"
REPO_DIR = "/content/vopt-learning"
PKGS = ["tensorrt-cu12", "onnx",]  # Colab 기본 torch/torchvision은 재설치하지 않음

assert os.path.isdir("/content"), "Colab 런타임이 아닙니다. 커널을 Colab으로 선택하세요."

def sh(cmd: str) -> None:
    print("$", cmd)
    subprocess.run(cmd, shell=True, check=True)

sh("pip install -q uv")
if os.path.isdir(REPO_DIR):
    sh(f"git -C {REPO_DIR} pull -q --ff-only")
else:
    sh(f"git clone -q --depth 1 {REPO_URL} {REPO_DIR}")
sh("uv pip install --system -q " + " ".join(PKGS))

if f"{REPO_DIR}/src" not in sys.path:
    sys.path.insert(0, f"{REPO_DIR}/src")

# autoreload: src/vopt 수정 사항을 커널 재시작 없이 반영
# Colab의 구버전 IPython autoreload는 Python 3.12+에서 삭제된 `imp`를 import하므로 최소한의 shim을 넣는다.
import importlib, types
if "imp" not in sys.modules:
    try:
        import imp  # noqa: F401  (Python <= 3.11)
    except ModuleNotFoundError:
        sys.modules["imp"] = types.SimpleNamespace(reload=importlib.reload)
try:
    get_ipython().run_line_magic("load_ext", "autoreload")
    get_ipython().run_line_magic("autoreload", "2")
except Exception as e:
    print(f"[warn] autoreload 비활성화 ({e!r}). vopt 수정 후에는 커널을 재시작하세요.")

from vopt.env import print_env
print_env()

$ pip install -q uv
$ git -C /content/vopt-learning pull -q --ff-only
$ uv pip install --system -q tensorrt-cu12 onnx
The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
python       : 3.13.16
torch        : 2.14.1+cu130
cuda (torch) : 13.0
gpu          : Tesla T4
gpu tag      : tesla-t4-sm75
tensorrt     : 11.4.0.106


In [2]:
import torch
import torchvision
from vopt import cache, data

B = 8
model = torchvision.models.resnet18(weights="DEFAULT").eval().cuda()
# 실제 사진 B장(Oxford-IIIT Pet, CC BY-SA 4.0. HF에서 필요한 장수만 받음)을 ResNet 표준 전처리한 [B, 3, 224, 224] 배치.
x = data.imagenet_batch(B)

# 최신 TensorRT는 strongly typed: engine 정밀도 = ONNX의 dtype.
# ONNX/engine 입력 shape은 export할 때의 x shape으로 고정되므로 파일명에 B를 넣는다.
onnx_path = cache.cache_dir() / f"resnet18_b{B}.onnx"
if not onnx_path.exists():
    torch.onnx.export(
        model, (x,), str(onnx_path),
        input_names=["input"], output_names=["logits"],
        opset_version=17, dynamo=False,
    )
print(onnx_path, f"{onnx_path.stat().st_size / 1e6:.1f} MB")

/content/vopt-cache/resnet18_b8.onnx 46.7 MB


## TensorRT 공통 함수

static engine과 dynamic engine 모두 똑같이 쓰는 부분만 함수로 묶습니다.

- `parse_onnx`: 빈 network를 만들고 ONNX를 parse해서 채운다.
- `build_and_save`: network + config로 engine을 빌드해서 파일로 저장한다.
- `load_engine`: engine 파일을 역직렬화하고 입출력 shape을 출력한다.

**둘이 달라지는 부분(builder config 설정)은 함수에 넣지 않고 각 build 셀에 그대로 둡니다.** 두 build 셀을 나란히 비교해 보세요.


In [ ]:
import tensorrt as trt
from vopt.trt_utils import PrintLogger

# 기본 trt.Logger는 C++ stderr로 출력해서 Colab 셀에 잘 안 보인다.
# TRT 경고/에러를 셀에서 보기 위해 print 로거를 쓴다.
# TensorRT는 프로세스당 logger 하나만 쓰므로(나중에 넘긴 다른 logger는 무시됨) 노트북 전체에서 이 logger 하나만 쓴다.
logger = PrintLogger()  # TRT 객체들이 쓰는 동안 살아 있어야 하므로 변수로 들고 있는다
runtime = trt.Runtime(logger)


def parse_onnx(builder, onnx_path):
    """빈 network를 만들고 ONNX 그래프로 채운다."""
    network = builder.create_network()  # TRT 11: strongly typed (레이어 정밀도 = ONNX dtype)
    parser = trt.OnnxParser(network, logger)
    with open(onnx_path, "rb") as f:
        if not parser.parse(f.read()):
            for i in range(parser.num_errors):
                print(parser.get_error(i))
            raise RuntimeError("ONNX 파싱 실패")

    for i in range(network.num_inputs):
        t = network.get_input(i)
        print(f"  network input : {t.name} {tuple(t.shape)}")  # 동적 축은 -1
    # network의 weight 메모리는 parser가 들고 있으므로, build가 끝날 때까지 parser도 살려 둬야 한다
    return network, parser


def build_and_save(builder, network, config, engine_path):
    """GPU에서 레이어별 kernel을 골라 engine을 빌드하고 파일로 저장한다."""
    engine_bytes = builder.build_serialized_network(network, config)
    if engine_bytes is None:
        raise RuntimeError("engine 빌드 실패")
    engine_path.write_bytes(engine_bytes)
    print(f"  saved : {engine_path} ({engine_path.stat().st_size / 1e6:.1f} MB)")


def load_engine(engine_path):
    """engine 파일을 역직렬화하고 입출력 텐서 정보를 출력한다."""
    with open(engine_path, "rb") as f:
        engine = runtime.deserialize_cuda_engine(f.read())
    assert engine is not None, "역직렬화 실패: 다른 GPU 아키텍처/TRT 버전에서 빌드된 engine일 수 있음"

    for i in range(engine.num_io_tensors):
        name = engine.get_tensor_name(i)
        print(f"{engine.get_tensor_mode(name)}  {name}  {engine.get_tensor_shape(name)}  {engine.get_tensor_dtype(name)}")
    return engine


ImportError: cannot import name 'PrintLogger' from 'vopt.trt_utils' (/content/vopt-learning/src/vopt/trt_utils.py)

In [ ]:
# ── static engine build ──
engine_path = cache.cache_dir() / f"resnet18_b{B}.engine"
if not engine_path.exists():
    builder = trt.Builder(logger)
    network, parser = parse_onnx(builder, onnx_path)

    config = builder.create_builder_config()
    config.set_memory_pool_limit(trt.MemoryPoolType.WORKSPACE, 1 << 30)
    # optimization profile 없음 → engine 입력 shape = ONNX에 고정된 [B, 3, 224, 224]

    build_and_save(builder, network, config, engine_path)

engine = load_engine(engine_path)  # static engine이면 shape에 -1(dynamic axis)이 없다
context = engine.create_execution_context()


NameError: name 'load_engine' is not defined

## 실험 1: 검증 없이 static engine에 batch 4 넣기

`set_input_shape`의 반환값을 확인하지 않는 추론 함수로 batch 4를 넣어 봅니다.

입력은 일부러 **8장 텐서 `x8`의 앞 4장 view(`x8[:4]`)**로 만듭니다. view는 `x8`과 메모리를 공유하므로 `x4.data_ptr() == x8.data_ptr()`이고, `x4` 바로 뒤에는 `x8`의 나머지 4장이 있습니다. 이렇게 하면

- TensorRT가 `x4` 범위 밖을 읽어도 그 자리는 실제로 할당된 메모리라서 crash가 날 수 없다.
- 밖에서 **무엇을** 읽었는지 알고 있으므로 결과로 증명할 수 있다.


In [ ]:
def trt_infer_unchecked(context, x):
    """set_input_shape의 반환값을 무시하는 (잘못된) 추론 함수."""
    x = x.contiguous()
    context.set_input_shape("input", tuple(x.shape))  # 실패해도 예외 없음, False만 반환하고 logger로 에러 출력

    out_shape = tuple(context.get_tensor_shape("logits"))
    out = torch.empty(out_shape, dtype=x.dtype, device="cuda")
    context.set_tensor_address("input", x.data_ptr())  # TensorRT는 주소만 받는다. 버퍼 크기는 모른다.
    context.set_tensor_address("logits", out.data_ptr())
    ok = context.execute_async_v3(torch.cuda.current_stream().cuda_stream)
    assert ok, "TensorRT 실행 실패"
    return out


In [ ]:
x8 = data.imagenet_batch(B)
x4 = x8[:B // 2]  # view: x8과 같은 메모리의 앞 절반
print(f"x4.shape: {tuple(x4.shape)},  x4.data_ptr() == x8.data_ptr(): {x4.data_ptr() == x8.data_ptr()}")

y8 = trt_infer_unchecked(context, x8)  # 정상 호출 (기준값)
y4 = trt_infer_unchecked(context, x4)  # batch 4를 넣었다. 에러가 날까?

print(f"\ny4.shape: {tuple(y4.shape)}  <- batch 4를 넣었는데 출력은 8개")
print(f"x4 크기: {x4.numel() * x4.element_size() / 1e6:.2f} MB,  engine이 읽은 크기: {x8.numel() * x8.element_size() / 1e6:.2f} MB")

# 앞 4개는 x4(=x8[:4])의 결과. 뒤 4개는 x4 밖에 있던 x8[4:]의 결과와 같아야 한다.
print(f"\nmax |y4[:4] - y8[:4]| = {(y4[:4] - y8[:4]).abs().max().item():.3g}   (x4 안의 이미지)")
print(f"max |y4[4:] - y8[4:]| = {(y4[4:] - y8[4:]).abs().max().item():.3g}   (x4 밖의 이미지를 읽었다는 증거)")


### 무슨 일이 일어났나

1. `set_input_shape("input", (4, 3, 224, 224))`는 static engine이 받을 수 없는 shape이라 **`False`를 반환**하고 logger에 에러를 남깁니다. 위의 `[TRT ERROR]` 줄이 그것입니다. **예외는 발생하지 않습니다.**
2. context의 입력 shape은 그대로 `[8, 3, 224, 224]`이고, 그래서 출력 shape도 `(8, 1000)`입니다.
3. TensorRT는 `x4.data_ptr()` 주소에서 **8장 분량**을 읽습니다. 뒤 4장은 `x4` 범위 밖, 여기서는 `x8[4:]`였기 때문에 `y4[4:]`가 `y8[4:]`와 같게 나왔습니다.

### 4장짜리 텐서를 따로 만들어 넣으면?

`data.imagenet_batch(4)`처럼 독립된 텐서를 넣어도 똑같이 8장 분량을 읽습니다. 다만 범위 밖에 무엇이 있는지는 알 수 없습니다(out-of-bounds read, undefined behavior).

- **대부분:** PyTorch caching allocator가 큰 segment 안에 텐서를 배치하므로, 범위 밖도 할당된 GPU 메모리입니다. 다른 텐서나 해제된 블록의 내용을 읽고 **에러 없이 틀린 출력**이 나옵니다.
- **운이 나쁘면:** 텐서가 segment 끝에 있으면 `CUDA error: an illegal memory access was encountered`가 납니다. 이 에러는 CUDA context에 남아서 **이후의 모든 CUDA 연산이 실패하고, 커널을 재시작해야** 합니다.

그래서 이 노트북은 안전한 view로 실험했습니다. 실제 코드에서는 반환값을 반드시 확인해야 합니다(실험 2).


## 실험 2: `set_input_shape` 반환값 확인하기

반환값이 `False`면 실행하기 전에 예외를 던집니다. 메모리를 건드리기 전에 막으므로 안전합니다.


In [ ]:
def trt_infer(context, x):
    x = x.contiguous()

    # 입력 shape을 context에 알려준다. engine이 허용하지 않는 shape이면 False를 반환한다.
    # 주의: TensorRT는 여기서 예외를 던지지 않는다(logger로 에러만 출력). 반환값을 직접 확인해야 한다.
    if not context.set_input_shape("input", tuple(x.shape)):
        shape = tuple(context.engine.get_tensor_shape("input"))
        if -1 in shape:  # dynamic engine: 허용 범위는 optimization profile의 min~max
            mn, _, mx = context.engine.get_tensor_profile_shape("input", 0)
            allowed = f"min {tuple(mn)} ~ max {tuple(mx)}"
        else:
            allowed = str(shape)
        raise ValueError(f"input shape {tuple(x.shape)}은 이 engine에서 실행할 수 없음 (허용: {allowed})")

    # 출력 shape은 입력 shape이 정해진 뒤에 context에서 얻는다 (dynamic engine이면 여기서 batch가 정해짐)
    out_shape = tuple(context.get_tensor_shape("logits"))
    out = torch.empty(out_shape, dtype=x.dtype, device="cuda")

    context.set_tensor_address("input", x.data_ptr())
    context.set_tensor_address("logits", out.data_ptr())
    ok = context.execute_async_v3(torch.cuda.current_stream().cuda_stream)
    assert ok, "TensorRT 실행 실패"
    return out


In [ ]:
print("=== Static engine에 다른 batch 넣기 (검증 있음) ===")
for inp in (x8, x4):
    print(f"input.shape: {tuple(inp.shape)}")
    try:
        out = trt_infer(context, inp)
        print(f"  -> out.shape: {tuple(out.shape)}")
    except ValueError as e:
        print(f"  -> Error occurred: {e}")


## 다양한 batch를 처리하는 dynamic engine

static engine은 build 때의 shape 하나만 받습니다. 여러 batch를 받으려면 **두 군데**를 바꿔야 합니다.

1. **ONNX export:** `dynamic_axes`로 batch 축(0번)을 동적으로 표시합니다. 그러면 ONNX 입력 shape이 `[batch_size, 3, 224, 224]`가 됩니다. TensorRT에서는 `-1`로 보입니다.
2. **TensorRT build:** **optimization profile**로 동적 축의 범위를 지정합니다.
   - `min` / `max`: 실행할 수 있는 shape의 범위. 이 밖의 shape은 `set_input_shape`이 `False`를 반환한다.
   - `opt`: kernel을 고를 때 기준으로 삼는 shape. 이 shape에서 가장 빠르고, 범위 안의 다른 shape도 실행은 된다.

ONNX만 dynamic이고 profile을 주지 않으면 build가 실패합니다. 반대로 ONNX가 static이면 profile을 줘도 범위를 바꿀 수 없습니다.


In [ ]:
DYN_MIN, DYN_OPT, DYN_MAX = 1, 8, 32

# ── 차이 ①: ONNX export에 dynamic_axes 추가 (나머지 인자는 static export와 같다) ──
dyn_onnx_path = cache.cache_dir() / "resnet18_dynamic.onnx"
if not dyn_onnx_path.exists():
    torch.onnx.export(
        model, (x,), str(dyn_onnx_path),
        input_names=["input"], output_names=["logits"],
        dynamic_axes={"input": {0: "batch_size"}, "logits": {0: "batch_size"}},  # ★ batch 축(0번)을 동적으로
        opset_version=17, dynamo=False,  # legacy exporter라 onnxscript 불필요
    )

# ── dynamic engine build ── static engine 파일을 덮어쓰지 않도록 경로를 분리한다.
dyn_engine_path = cache.cache_dir() / f"resnet18_dynamic_b{DYN_MIN}-{DYN_OPT}-{DYN_MAX}.engine"
if not dyn_engine_path.exists():
    builder = trt.Builder(logger)
    network, parser = parse_onnx(builder, dyn_onnx_path)  # network input이 (-1, 3, 224, 224)로 보인다

    config = builder.create_builder_config()
    config.set_memory_pool_limit(trt.MemoryPoolType.WORKSPACE, 1 << 30)

    # ── 차이 ②: optimization profile로 동적 축의 범위를 지정 ──
    profile = builder.create_optimization_profile()
    profile.set_shape(
        "input",
        min=(DYN_MIN, 3, 224, 224),  # 허용 최소 shape
        opt=(DYN_OPT, 3, 224, 224),  # kernel 선택 기준 shape (가장 빠름)
        max=(DYN_MAX, 3, 224, 224),  # 허용 최대 shape
    )
    config.add_optimization_profile(profile)

    build_and_save(builder, network, config, dyn_engine_path)


In [ ]:
# static engine/context는 그대로 두고 dyn_ 접두사로 구분한다.
dyn_engine = load_engine(dyn_engine_path)  # dynamic engine은 동적 축이 -1로 보인다

# 실제 허용 범위는 profile에 있다
mn, opt, mx = dyn_engine.get_tensor_profile_shape("input", 0)
print(f"profile 0 (input): min {tuple(mn)}, opt {tuple(opt)}, max {tuple(mx)}")

dyn_context = dyn_engine.create_execution_context()


In [ ]:
x_max = data.imagenet_batch(DYN_MAX)  # 이미지 DYN_MAX장. 아래에서 앞 b장씩 잘라 쓴다.

print("=== Dynamic engine에 여러 batch 넣기 (PyTorch 출력과 비교) ===")
for b in (DYN_MIN, 3, DYN_OPT, DYN_MAX):
    xb = x_max[:b]
    y_trt = trt_infer(dyn_context, xb)
    with torch.inference_mode():
        y_torch = model(xb)
    top1 = (y_trt.argmax(-1) == y_torch.argmax(-1)).float().mean().item() * 100
    print(f"batch {b:2d}: out.shape {tuple(y_trt.shape)},  max |diff| {(y_trt - y_torch).abs().max().item():.2e},  top-1 일치 {top1:.0f}%")

print("\n=== profile 범위 밖 batch ===")
x_over = torch.randn(DYN_MAX + 1, 3, 224, 224, device="cuda")  # 검증에서 막히므로 내용은 상관없음
try:
    trt_infer(dyn_context, x_over)
except ValueError as e:
    print(f"batch {DYN_MAX + 1}: Error occurred: {e}")


### 정리

- dynamic engine에서는 `set_input_shape`이 실제로 context의 shape을 바꾸고, `get_tensor_shape("logits")`도 그 batch에 맞춰 나옵니다. 그래서 batch 1~32 모두 출력 shape이 입력과 같습니다.
- profile 범위(`max` 32)를 넘는 batch는 static engine 때와 마찬가지로 `set_input_shape`이 `False`를 반환합니다. 반환값 확인은 dynamic engine에서도 필요합니다.
- kernel은 `opt` shape 기준으로 골랐기 때문에, 같은 batch라면 static engine보다 약간 느릴 수 있습니다. 주로 쓰는 batch를 `opt`로 두세요.

> 실행 중 `Using default stream in enqueueV3() may lead to performance issues` 경고가 보일 수 있습니다. torch의 기본 stream에서 실행해서 TensorRT가 추가로 동기화한다는 성능 경고이고, 결과에는 영향이 없습니다.
